In [1]:
import pandas as pd
import numpy as np
from tabulate import tabulate
import sys
from IPython.core.interactiveshell import InteractiveShell
import holidays

# 행, 열 무제한 출력
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 셀 너비 무제한
pd.set_option('display.max_colwidth', None)

# 파이썬 기본 출력 길이 무제한
np.set_printoptions(threshold=sys.maxsize)

# 여러 출력 값을 전부 표시
InteractiveShell.ast_node_interactivity = "all"

# 데이터 확인  

In [2]:
def check_data(path):
    df = pd.read_csv(path, encoding='utf-8')
    print('데이터 컬럼 확인')
    print(df.columns, '\n')
    print('데이터 상위 5개 추출')
    print(tabulate(df.head(),headers=df.columns), '\n')
    print('데이터 타입 확인')
    print(df.info())
    print('데이터 Nan값 개수 확인')
    print(df.isna().sum())


## building_info 데이터 확인  

- 건물번호: 각 건물의 고유 식별번호  
- 건물유형: 건물의 용도별 분류 (호텔, 상용, 병원, 학교 등)  
- 연면적(m²): 건물 전체 바닥면적 (m2)  
- 냉방면적(m²): 공조시스템이 설치된 면적 (m2)  
- 태양광용량(kW): 설치된 태양광 발전 시설의 용량 (kWh)  
- ESS저장용량(kWh): 에너지 저장 시스템(Energy Storage System) 용량 (kWh)  
- PCS용량(kW): 전력변환시스템(Power Conversion System) 용량 (kW)  

In [3]:
data_path = 'datas/building_info.csv' 
check_data(data_path)
building_info_df = pd.read_csv(data_path)
building_info_df.head().to_csv('temp.csv', encoding='utf-8')

데이터 컬럼 확인
Index(['건물번호', '건물유형', '연면적(m2)', '냉방면적(m2)', '태양광용량(kW)', 'ESS저장용량(kWh)',
       'PCS용량(kW)'],
      dtype='object') 

데이터 상위 5개 추출
      건물번호  건물유형      연면적(m2)    냉방면적(m2)  태양광용량(kW)    ESS저장용량(kWh)    PCS용량(kW)
--  ----------  ----------  ------------  --------------  ----------------  ------------------  -------------
 0           1  호텔             82912.7         77586    -                 -                   -
 1           2  상용             40658.9         30392.8  -                 -                   -
 2           3  병원            560431          418992    278.58            -                   -
 3           4  호텔             41813.3         23715.7  -                 -                   -
 4           5  학교            403749          248507    1983.05           1025                250 

데이터 타입 확인
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        ------

## train 데이터 확인  

- num_date_time: 고유 식별자 (건물번호_날짜시간 형식)  
- 건물번호: 건물 식별 번호  
- 일시: 측정 날짜와 시간 (YYYYMMDD HH 형식)  
- 기온: 외부 기온 (섭씨)  
- 강수량: 시간당 강수량   
- 풍속: 풍속   
- 습도: 상대습도   
- 일조: 일조시간   
- 일사(MJ/m²): 일사량   
- 전력소비량(kWh): 건물의 시간별 전력 소비량 (kWh)  

In [4]:
data_path = 'datas/train.csv'
check_data(data_path)
train_df = pd.read_csv(data_path)

데이터 컬럼 확인
Index(['num_date_time', '건물번호', '일시', '기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)',
       '일조(hr)', '일사(MJ/m2)', '전력소비량(kWh)'],
      dtype='object') 

데이터 상위 5개 추출
    num_date_time      건물번호  일시           기온(°C)    강수량(mm)    풍속(m/s)    습도(%)    일조(hr)    일사(MJ/m2)    전력소비량(kWh)
--  ---------------  ----------  -----------  ----------  ------------  -----------  ---------  ----------  -------------  -----------------
 0  1_20240601 00             1  20240601 00        18.3             0          2.6         82           0              0            5794.8
 1  1_20240601 01             1  20240601 01        18.3             0          2.7         82           0              0            5591.85
 2  1_20240601 02             1  20240601 02        18.1             0          2.6         80           0              0            5338.17
 3  1_20240601 03             1  20240601 03        18               0          2.6         81           0              0            4554.42
 4  1_20

In [5]:
# 열 수정
train_df.columns = ['num_date_time', '건물번호', '일시', '기온', '강수량', '풍속', '습도', '일조', '일사', '전력소비량']

In [6]:
# 일시에서 날짜와 시간 추출
train_df['시간'] = train_df['일시'].str.split(' ').str[1]
train_df['날짜'] = train_df['일시'].str.split(' ').str[0]

train_df['연'] = train_df['날짜'].str[:4]
train_df['월'] = train_df['날짜'].str[4:6]
train_df['일'] = train_df['날짜'].str[6:]
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01


In [7]:
# 계절 생성
train_df['월_int'] = train_df['월'].astype(int)

condition = [
    train_df['월_int'].isin([3,4,5]),
    train_df['월_int'].isin([6,7,8]),
    train_df['월_int'].isin([9,10,11]),
    train_df['월_int'].isin([12,1,2])
]

choice  = ['봄', '여름', '가을', '겨울']
train_df['계절'] = np.select(condition, choice, default='알수없음')
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,월_int,계절
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01,6,여름
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01,6,여름
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01,6,여름
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01,6,여름
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01,6,여름


In [8]:
# 휴일 데이터 추가

# 한국 공휴일 객체 생성 (year 지정 가능)
kr_holidays = holidays.KR()

train_df['날짜2'] = pd.to_datetime(train_df['날짜'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
train_df['휴일'] = train_df['날짜2'].apply(lambda x: True if x in kr_holidays else False)

train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,월_int,계절,날짜2,휴일
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01,6,여름,2024-06-01,False
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01,6,여름,2024-06-01,False
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01,6,여름,2024-06-01,False
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01,6,여름,2024-06-01,False
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01,6,여름,2024-06-01,False


In [9]:
def korean_apparent_temperature(T, RH, V):
    """
    한국기상청 체감온도 공식 (모든 계절 적용)
    
    Parameters:
    T: 기온
    RH: 상대습도
    V: 풍속
    
    Returns:
    체감온도
    """
    # 여름철 체감온도 (기온 > 20°C, 높은 습도 환경)
    if T > 20:
        # Heat Index 기반 한국형 공식
        AT = 1.07 * T + 0.2 * RH - 0.65 * V - 2.70
    
    # 겨울철 체감온도 (기온 ≤ 10°C, 바람의 영향 고려)
    elif T <= 10:
        V_kmh = V * 3.6  # m/s를 km/h로 변환
        
        # 바람이 약할 때는 기온과 동일
        if V_kmh < 4.8:
            AT = T
        else:
            # Wind Chill 공식 적용
            AT = 13.12 + 0.6215*T - 11.37*(V_kmh**0.16) + 0.3965*T*(V_kmh**0.16)
    
    # 중간 계절 (10°C < 기온 ≤ 20°C)
    else:
        # 기온, 습도, 풍속을 모두 고려한 절충 공식
        # 습도 효과는 줄이고, 풍속 효과는 적당히 적용
        humidity_effect = (RH - 50) * 0.1 if RH > 50 else 0
        wind_effect = V * 0.5
        AT = T + humidity_effect - wind_effect
    
    return round(AT, 2)

In [10]:
def calculate_apparent_temp_features(df, temp_col='기온', humidity_col='습도', wind_col='풍속'):
    """
    데이터프레임에 체감온도 관련 특성들을 추가하는 함수
    
    Parameters:
    df: 입력 데이터프레임
    temp_col: 기온 컬럼명
    humidity_col: 습도 컬럼명  
    wind_col: 풍속 컬럼명
    
    Returns:
    체감온도 특성이 추가된 데이터프레임
    """
    df = df.copy()
    
    # 체감온도 계산
    df['체감온도'] = df.apply(lambda row: korean_apparent_temperature(
        row[temp_col], 
        row[humidity_col], 
        row[wind_col]
    ), axis=1)
    
    # 추가 특성들
    df['체감온도_차이'] = df['체감온도'] - df[temp_col]
    df['체감온도_비율'] = df['체감온도'] / df[temp_col]
    
    # 체감온도 범주화
    def categorize_apparent_temp(temp):
        if temp <= 0:
            return '매우추움'
        elif temp <= 10:
            return '추움'
        elif temp <= 20:
            return '서늘'
        elif temp <= 25:
            return '적당'
        elif temp <= 30:
            return '더움'
        else:
            return '매우더움'
    
    df['체감온도_범주'] = df['체감온도'].apply(categorize_apparent_temp)
    
    # 불쾌지수 계산 (추가 특성)
    df['불쾌지수'] = 0.81 * df[temp_col] + 0.01 * df[humidity_col] * (0.99 * df[temp_col] - 14.3) + 46.3
    
    return df


In [11]:
train_df = calculate_apparent_temp_features(train_df)
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,월_int,계절,날짜2,휴일,체감온도,체감온도_차이,체감온도_비율,체감온도_범주,불쾌지수
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01,6,여름,2024-06-01,False,20.20,1.90,1.103825,적당,64.25294
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01,6,여름,2024-06-01,False,20.15,1.85,1.101093,적당,64.25294
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.70,1.093923,서늘,63.85620
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.80,1.100000,서늘,63.73120
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01,6,여름,2024-06-01,False,20.25,2.45,1.137640,적당,63.40882


In [12]:
def calculate_simple_degree_days(df, temp_col='기온', season_col='계절'):
    """
    계절별 기준온도를 적용한 간단한 CDD/HDD 계산
    
    Parameters:
    df: 데이터프레임 (계절 컬럼이 있어야 함)
    temp_col: 기온 컬럼명
    season_col: 계절 컬럼명
    
    Returns:
    CDD, HDD 컬럼이 추가된 데이터프레임
    """
    df = df.copy()
    
    # 계절별 기준온도 정의 (한국 기준, 호텔 건물용)
    season_base_temps = {
        '겨울': {'cooling': 22, 'heating': 20},  # 겨울: 난방 위주
        '봄': {'cooling': 24, 'heating': 18},    # 봄: 중간 계절
        '여름': {'cooling': 26, 'heating': 16},  # 여름: 냉방 위주  
        '가을': {'cooling': 24, 'heating': 18}   # 가을: 중간 계절
    }
    
    # CDD, HDD 컬럼 초기화
    df['CDD'] = 0.0
    df['HDD'] = 0.0
    
    # 계절별로 계산
    for season in season_base_temps.keys():
        mask = df[season_col] == season
        if mask.any():
            cooling_base = season_base_temps[season]['cooling']
            heating_base = season_base_temps[season]['heating']
            
            # 냉방도일: max(0, 기온 - 냉방기준온도)
            df.loc[mask, 'CDD'] = np.maximum(0, df.loc[mask, temp_col] - cooling_base)
            
            # 난방도일: max(0, 난방기준온도 - 기온)  
            df.loc[mask, 'HDD'] = np.maximum(0, heating_base - df.loc[mask, temp_col])
    
    # 소수점 둘째자리까지 반올림
    df['CDD'] = df['CDD'].round(2)
    df['HDD'] = df['HDD'].round(2)
    
    return df

def add_degree_days_features(df, temp_col='기온', season_col='계절'):
    """
    도일 관련 추가 특성들도 함께 생성
    """
    df = calculate_simple_degree_days(df, temp_col, season_col)
    
    # 추가 특성들
    df['총도일'] = df['CDD'] + df['HDD']  # 총 도일
    df['냉난방구분'] = np.where(df['CDD'] > df['HDD'], '냉방', '난방')  # 주요 부하
    df['냉난방구분'] = np.where((df['CDD'] == 0) & (df['HDD'] == 0), '중립', df['냉난방구분'])
    
    return df

In [13]:
train_df = calculate_simple_degree_days(train_df)
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,월_int,계절,날짜2,휴일,체감온도,체감온도_차이,체감온도_비율,체감온도_범주,불쾌지수,CDD,HDD
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01,6,여름,2024-06-01,False,20.20,1.90,1.103825,적당,64.25294,0.0,0.0
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01,6,여름,2024-06-01,False,20.15,1.85,1.101093,적당,64.25294,0.0,0.0
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.70,1.093923,서늘,63.85620,0.0,0.0
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.80,1.100000,서늘,63.73120,0.0,0.0
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01,6,여름,2024-06-01,False,20.25,2.45,1.137640,적당,63.40882,0.0,0.0


In [14]:
def classify_weather_category(df, 
                             rain_col='강수량', 
                             sunshine_col='일조', 
                             solar_col='일사',
                             humidity_col='습도'):
    """
    날씨 카테고리 분류 함수
    
    Parameters:
    df: 데이터프레임
    rain_col: 강수량 컬럼명
    sunshine_col: 일조시간 컬럼명  
    solar_col: 일사량 컬럼명
    humidity_col: 습도 컬럼명
    
    Returns:
    날씨카테고리가 추가된 데이터프레임
    """
    df = df.copy()
    
    def categorize_weather(row):
        rain = row[rain_col]
        sunshine = row[sunshine_col]
        solar = row[solar_col]
        humidity = row[humidity_col]
        
        # 1. 강수 여부 우선 확인
        if rain > 5.0:  # 5mm 이상
            return '비'
        elif rain > 0.5:  # 0.5~5mm
            return '소나기'
        
        # 2. 강수가 없는 경우 일조/일사량으로 판단
        elif sunshine > 0.8:  # 1시간 중 48분 이상 햇빛
            if solar > 2.0:
                return '맑음'
            else:
                return '약간맑음'
        
        elif sunshine > 0.3:  # 1시간 중 18분 이상 햇빛
            return '구름조금'
            
        elif sunshine > 0.0:  # 조금이라도 햇빛
            return '구름많음'
            
        # 3. 일조가 전혀 없는 경우 습도로 세분화
        else:
            if humidity >= 85:
                if rain > 0:
                    return '이슬비'
                else:
                    return '안개'
            elif humidity >= 75:
                return '흐림'
            else:
                return '구름많음'
    
    # 날씨 카테고리 적용
    df['날씨카테고리'] = df.apply(categorize_weather, axis=1)
    
    return df

def classify_detailed_weather_category(df,
                                     rain_col='강수량', 
                                     sunshine_col='일조',
                                     solar_col='일사',
                                     humidity_col='습도',
                                     wind_col='풍속',
                                     temp_col='기온'):
    """
    더 세분화된 날씨 카테고리 분류
    """
    df = df.copy()
    
    def detailed_categorize_weather(row):
        rain = row[rain_col]
        sunshine = row[sunshine_col] 
        solar = row[solar_col]
        humidity = row[humidity_col]
        wind = row[wind_col]
        temp = row[temp_col]
        
        # 강수 기준 분류
        if rain >= 20:
            return '폭우'
        elif rain >= 10:
            return '강한비'
        elif rain >= 3:
            return '비'
        elif rain > 0.5:
            return '약한비'
        elif rain > 0:
            return '이슬비'
        
        # 맑음 정도 분류 (강수 없을 때)
        elif sunshine >= 0.9 and solar >= 3.0:
            return '매우맑음'
        elif sunshine >= 0.7 and solar >= 2.0:
            return '맑음'
        elif sunshine >= 0.4:
            return '약간맑음'
        elif sunshine > 0:
            return '구름조금'
        
        # 흐림 정도 분류 (일조 없을 때)
        else:
            if humidity >= 90:
                if temp < 5:  # 겨울철 저온
                    return '서리'
                else:
                    return '안개'
            elif humidity >= 80:
                if wind >= 4:  # 바람 강함
                    return '흐리고바람'
                else:
                    return '흐림'
            elif humidity >= 70:
                return '구름많음'
            else:
                return '건조한흐림'
    
    # 세분화된 날씨 카테고리 적용
    df['상세날씨카테고리'] = df.apply(detailed_categorize_weather, axis=1)
    
    return df

def add_weather_numeric_features(df,
                                rain_col='강수량', 
                                sunshine_col='일조', 
                                solar_col='일사',
                                humidity_col='습도',
                                weather_cat_col='날씨카테고리'):
    """
    날씨 카테고리를 기반으로 한 수치형 특성 추가
    """
    df = df.copy()
    
    # 맑음 지수 (0~1)
    df['맑음지수'] = (df[sunshine_col] * 0.6 + 
                    np.minimum(df[solar_col]/4, 1) * 0.4)
    
    # 습윤 지수 (0~1)  
    df['습윤지수'] = (df[rain_col]/10 * 0.7 + 
                    (df[humidity_col]-30)/70 * 0.3).clip(0, 1)
    
    # 날씨 점수 (맑을수록 높음)
    weather_scores = {
        '매우맑음': 10, '맑음': 8, '약간맑음': 6, '구름조금': 5,
        '구름많음': 4, '흐림': 3, '약한비': 2, '비': 1, '강한비': 0
    }
    
    df['날씨점수'] = df[weather_cat_col].map(weather_scores).fillna(3)
    
    # 이진 특성들
    df['맑은날'] = (df[weather_cat_col].isin(['매우맑음', '맑음', '약간맑음'])).astype(int)
    df['비오는날'] = (df[weather_cat_col].str.contains('비')).astype(int)
    df['흐린날'] = (df[weather_cat_col].isin(['흐림', '구름많음', '안개'])).astype(int)
    
    return df

In [15]:
print("=== 기본 날씨 카테고리 분류 ===")
train_df = classify_weather_category(train_df)
# print(train_df[['강수량', '일조', '일사', '습도', '날씨카테고리']])

print("\n=== 상세 날씨 카테고리 분류 ===") 
train_df = classify_detailed_weather_category(train_df)
# print(train_df[['강수량', '일조', '습도', '기온', '상세날씨카테고리']])

print("\n=== 날씨 수치형 특성 ===")
train_df = add_weather_numeric_features(train_df)
# print(train_df[['날씨카테고리', '맑음지수', '습윤지수', '날씨점수', '맑은날', '비오는날']])

train_df.head()

=== 기본 날씨 카테고리 분류 ===

=== 상세 날씨 카테고리 분류 ===

=== 날씨 수치형 특성 ===


,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,월_int,계절,날짜2,휴일,체감온도,체감온도_차이,체감온도_비율,체감온도_범주,불쾌지수,CDD,HDD,날씨카테고리,상세날씨카테고리,맑음지수,습윤지수,날씨점수,맑은날,비오는날,흐린날
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,20240601,2024,06,01,6,여름,2024-06-01,False,20.20,1.90,1.103825,적당,64.25294,0.0,0.0,흐림,흐림,0.0,0.222857,3.0,0,0,1
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,20240601,2024,06,01,6,여름,2024-06-01,False,20.15,1.85,1.101093,적당,64.25294,0.0,0.0,흐림,흐림,0.0,0.222857,3.0,0,0,1
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.70,1.093923,서늘,63.85620,0.0,0.0,흐림,흐림,0.0,0.214286,3.0,0,0,1
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,20240601,2024,06,01,6,여름,2024-06-01,False,19.80,1.80,1.100000,서늘,63.73120,0.0,0.0,흐림,흐림,0.0,0.218571,3.0,0,0,1
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,20240601,2024,06,01,6,여름,2024-06-01,False,20.25,2.45,1.137640,적당,63.40882,0.0,0.0,흐림,흐림,0.0,0.218571,3.0,0,0,1


In [16]:
train_df[['날씨카테고리', '상세날씨카테고리']].head()

,날씨카테고리,상세날씨카테고리
0,흐림,흐림
1,흐림,흐림
2,흐림,흐림
3,흐림,흐림
4,흐림,흐림
